# Точность врёт: precision, recall, F1 и честный выбор k

Сервис отдельно платит за поиск восьмёрок в индексах. Здесь доля восьмёрок мала — и обычная точность перестаёт работать.

In [ ]:
from pathlib import Path
import pandas as pd


def find_digits_csv() -> Path:
    for p in (Path('digits.csv'), Path('../../data/digits.csv'), Path('../data/digits.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('digits.csv не найден — положите файл рядом с ноутбуком')


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score


## 1. Задача с двумя ответами

Постройте столбец `is_eight` (1, если цифра 8, иначе 0) и его долю -> `share_8`.

In [ ]:
is_eight = None
share_8 = None
assert is_eight is not None and share_8 is not None
assert set(pd.unique(is_eight)) == {0, 1}
assert 0.05 < float(share_8) < 0.15
print(round(float(share_8), 4))

## 2. Ленивый ответ «это не восьмёрка»

Какова точность правила, которое **всегда** отвечает 0? -> `acc_dumb`.

В `WHY_ACC_LIES` объясните, почему сервис не примет такой распознаватель, хотя точность высокая.

In [ ]:
acc_dumb = None
WHY_ACC_LIES = ''
assert acc_dumb is not None and float(acc_dumb) > 0.85
assert len(WHY_ACC_LIES) > 60
print(round(float(acc_dumb), 4), WHY_ACC_LIES)

## 3. Четыре числа вместо одного

Разбейте данные (`test_size=0.25`, `random_state=0`, `stratify=is_eight`), обучите kNN k=3 и посчитайте **вручную**:

- `tp` — предсказали 8 и это 8;
- `fp` — предсказали 8, а это не 8;
- `fn` — пропустили настоящую 8;
- `tn` — верно сказали «не 8».

**How:** сравнение массивов предсказаний и ответов, как `confusion_counts` в модуле 1.

In [ ]:
tp = fp = fn = tn = None
assert None not in (tp, fp, fn, tn)
assert tp + fp + fn + tn == 450
assert int(tp) > 0
print(tp, fp, fn, tn)

## 4. precision, recall, F1 руками

- `precision` = tp / (tp + fp) — какая доля наших «восьмёрок» настоящие;
- `recall` = tp / (tp + fn) — какую долю настоящих восьмёрок мы нашли;
- `f1_manual` = 2·precision·recall / (precision + recall).

Сверьте с `f1_score(y_te, pred)` -> `f1_lib`.

**Вопрос:** какая из двух ошибок дороже для почтового сервиса?

In [ ]:
precision = recall = f1_manual = None
f1_lib = None
assert None not in (precision, recall, f1_manual, f1_lib)
assert abs(float(f1_manual) - float(f1_lib)) < 1e-9
print(round(float(precision), 4), round(float(recall), 4), round(float(f1_manual), 4))

## 5. Третья часть данных: где выбирать k

Вернёмся к задаче «какая это цифра». Разбейте таблицу на три части: **обучающую** 60%, **проверочную** 20%, **финальную** 20% (`stratify` на каждом шаге).

Для k из 1, 3, 5, 7, 9, 15 посчитайте точность на **проверочной** части -> `val_table` (столбцы `k`, `accuracy`), лучшее k -> `best_k`.

**Правило:** финальную часть на этом шаге не трогаем.

In [ ]:
X_fit = X_val = X_final = None
y_fit = y_val = y_final = None
val_table = None
best_k = None
assert X_fit is not None and len(X_fit) > 1000
assert len(X_val) == len(X_final) == 360
assert val_table is not None and len(val_table) == 6
assert list(val_table.columns) == ['k', 'accuracy']
assert int(best_k) in (1, 3, 5, 7, 9, 15)
print(val_table, best_k)

## 6. Один выстрел по финальной части

Обучите kNN с `best_k` и посчитайте точность на финальной части -> `acc_final`.

В `WHY_ONE_SHOT` объясните, почему это число можно получить только один раз и почему оно обычно чуть ниже проверочного.

In [ ]:
acc_final = None
WHY_ONE_SHOT = ''
assert acc_final is not None and float(acc_final) > 0.9
assert len(WHY_ONE_SHOT) > 80
print(round(float(acc_final), 4), WHY_ONE_SHOT)

## 7. Опровержение: «F1 всегда лучше accuracy»

Посчитайте `f1_micro = f1_score(y_final, pred_final, average='micro')` и сравните с `acc_final`.

В `F1_LIMIT` объясните, что показал этот опыт и когда F1 действительно нужен.

In [ ]:
f1_micro = None
F1_LIMIT = ''
assert f1_micro is not None
assert abs(float(f1_micro) - float(acc_final)) < 1e-9
assert len(F1_LIMIT) > 60
print(round(float(f1_micro), 4), F1_LIMIT)

## 8. Расширение: F1 по каждой цифре

`f1_macro` = `f1_score(..., average='macro')` — среднее F1 по десяти цифрам. Сравните с `f1_micro`.

В `MACRO_NOTE` — в какой ситуации macro будет заметно ниже micro. **Готового ответа нет.**

In [ ]:
f1_macro = None
MACRO_NOTE = ''
assert f1_macro is not None and 0.5 < float(f1_macro) <= 1.0
assert len(MACRO_NOTE) > 50
print(round(float(f1_macro), 4), MACRO_NOTE)